# Глубинное обучение для текстовых данных, ФКН ВШЭ
## Домашнее задание 3: Уменьшение размеров модели
### Оценивание и штрафы

Максимально допустимая оценка за работу — __16 баллов__.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Весь код должен быть написан самостоятельно. Чужим кодом для пользоваться запрещается даже с указанием ссылки на источник. В разумных рамках, конечно. Взять пару очевидных строчек кода для реализации какого-то небольшого функционала можно.

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

__Мягкий дедлайн 22.11.26 23:59__ \
__Жесткий дедлайн 24.11.26 23:59__

### О задании

В этом задании вам предстоит научиться решать задачу Named Entity Recognition (NER) на самом популярном датасете – [CoNLL-2003](https://paperswithcode.com/dataset/conll-2003). В вашем распоряжении будет предобученный BERT, который вам необходимо уменьшить с минимальными потерями в качестве до размера 20М параметров. Для этого вы самостоятельно реализуете факторизацию эмбеддингов, дистилляцию, шеринг параметров и так далее.

В этом задании вам придется проводить довольно много экспериментов, поэтому мы рекомендуем не писать весь код в тетрадке, а завести разные файлы для отдельных логических блоков и скомпоновать все в виде проекта. Это позволит вашему ноутбуку не разрастаться и сильно облегчит задачу и вам, и проверяющим. Так же постарайтесь логгировать все ваши эксперименты на wandb или других платформах, чтобы ничего не потерялось.

В этом задании __запрещается использовать агентов для написания кода__, но разрешается для помощи в чтении статей и осознании методов.

### Оценивание
Оценка за это домашнее задание будет формироваться из оценки за __задания__ и за __отчет__, в котором от вас требуется написать о проделанной работе. За отчет можно получить до 2-х баллов, однако в случае отсутствия отчета баллы за соответствующие задания не будут ставиться. Задания делятся на две части: _номерные_ и _на выбор_. За _номерные_ можно получить в сумме 6 баллов, за задания _на выбор_ можно получить до 8. То есть за все дз можно получить 16 баллов (но не радуйтесь рано, это не так просто). Все, что вы наберете свыше 10, будет считаться бонусами.

### О датасете

Named Entity Recognition – это задача классификации токенов по классам сущностей. В CoNLL-2003 для именования сущностей используется маркировка **BIO** (Beggining, Inside, Outside), в которой метки означают следующее:

- *B-{метка}* – начало сущности *{метка}*
- *I-{метка}* – продолжнение сущности *{метка}*
- *O* – не сущность

Существуют так же и другие способы маркировки, например, BILUO. Почитать о них можно [тут](https://en.wikipedia.org/wiki/Inside–outside–beginning_(tagging)) и [тут](https://www.youtube.com/watch?v=dQw4w9WgXcQ).

Всего в датасете есть 9 разных меток.
- O – слову не соответствует ни одна сущность.
- B-PER/I-PER – слово или набор слов соответстует определенному _человеку_.
- B-ORG/I-ORG – слово или набор слов соответстует определенной _организации_.
- B-LOC/I-LOC – слово или набор слов соответстует определенной _локации_.
- B-MISC/I-MISC – слово или набор слов соответстует сущности, которая не относится ни к одной из предыдущих. Например, национальность, произведение искусства, мероприятие и т.д.

Приступим!

Начнем с загрузки и предобработки датасета.

In [ ]:
import json
import math
import site
import sys
from pathlib import Path

site.addsitedir(site.getusersitepackages())
ROOT = Path.cwd().resolve()
if ROOT.name == 'hw3_distillation_work':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'hw3_distillation_work'))

import torch
import torch.nn.functional as F
from datasets import load_dataset
from torch import nn
from torch.optim import AdamW
from transformers import AutoTokenizer, BertConfig, BertForTokenClassification, get_linear_schedule_with_warmup
import experiment as e
from experiment import LABELS, BASE, DATA, OUTPUT, DEVICE, evaluate, loaders, load_state, seed_everything


In [ ]:
print(Path.cwd())


In [ ]:
files = {
    split: str(DATA / f'{split}-00000-of-00001.parquet')
    for split in ('train', 'validation', 'test')
}
raw_dataset = load_dataset('parquet', data_files=files)
dataset = raw_dataset.remove_columns(['id', 'pos_tags', 'chunk_tags'])
dataset


In [ ]:
{name: len(split) for name, split in dataset.items()}


In [ ]:
dataset['train'][1]

In [ ]:
dataset['train'][0]

In [ ]:
label_names = LABELS


In [ ]:
words = dataset["train"][0]["tokens"]
labels = dataset["train"][0]["ner_tags"]

for i in range(len(words)):
    print(f'{words[i]}\t{label_names[labels[i]]}')

### Предобработка

На протяжении всего домашнего задания мы будем использовать _cased_ версию BERT, то есть токенизатор будет учитывать регистр слов. Для задачи NER регистр важен, так как имена и названия организаций или предметов искусства часто пишутся с большой буквы, и будет глупо прятать от модели такую информацию.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE, local_files_only=True, use_fast=True)


При токенизации слова могут разделиться на несколько токенов (как слово `Fischler` из примера ниже), из-за чего появится несоответствие между числом токенов и меток. Это несоответствие нам придется устранить вручную.

In [ ]:
example = dataset["train"][12]
words = example["tokens"]
tags = [label_names[t] for t in example["ner_tags"]]
tokenized_text = tokenizer(example["tokens"], is_split_into_words=True)

print('Слова: ', words)
print('Токены:', tokenized_text.tokens())
print('Метки:', tags)

__Задание 1 (1 балл).__ Токенизируйте весь датасет и для каждого текста выравните токены с метками так, чтобы каждому токену соответствовала одна метка. При этом важно сохранить нотацию BIO. И не забудьте про специальные токены! Должно получиться что-то такое:

In [ ]:
def align_labels_with_tokens(labels, tokenized):
    result = []
    previous_word = None
    word_ids = tokenized.word_ids
    for word in word_ids() if callable(word_ids) else word_ids:
        if word is None:
            result.append(-100)
        else:
            label = labels[word]
            if word == previous_word and label > 0 and label % 2 == 1:
                label += 1
            result.append(label)
        previous_word = word
    return result


aligned_labels = align_labels_with_tokens(example['ner_tags'], tokenized_text)
tags = [label_names[t] if t > -1 else t for t in aligned_labels]
print('Выровненные метки:', aligned_labels)
print('Выровненные названия меток:', tags)


In [ ]:
def tokenize_batch(batch):
    encoded = tokenizer(batch['tokens'], is_split_into_words=True, truncation=True, max_length=512)
    encoded['labels'] = [
        align_labels_with_tokens(tags, encoded[i])
        for i, tags in enumerate(batch['ner_tags'])
    ]
    return encoded

tokenized_dataset = raw_dataset.map(
    tokenize_batch, batched=True, remove_columns=raw_dataset['train'].column_names
)
tokenized_dataset


### Метрика

Для оценки качества NER обычно используют F1 меру с микро-усреднением. Мы загрузим ее из библиотеки `seqeval`. Функция `f1_score` принимает два 2d списка с правильными и предсказанными метками, записаными текстом, и возвращает для них значение F1. Вы можете использовать ее с параметрами по умолчанию.

In [ ]:
from seqeval.metrics import f1_score


In [ ]:
from seqeval.metrics import f1_score

Особенность подсчета F1 для NER заключается в том, что в некоторых ситуациях неправильные ответы могут засчитываться как правильные. Например, если модель предсказала `['I-PER', 'I-PER']`, то мы можем догадаться, что на самом деле должно быть `['B-PER', 'I-PER']`, так как сущность не может начинаться с `I-`. Функция `f1_score` учитывает это и поэтому работает только с текстовыми представлениями меток.

### Модель

В качестве базовой модели мы возьмем `bert-base-cased`. Как вы понимаете, он не обучался на задачу NER. Поэтому прежде чем приступать к уменьшению размера BERT, его необходимо дообучить.

__Задание 2 (1 балл)__ Дообучите `bert-base-cased` на нашем датасете с помощью обычного fine-tuning. У вас должно получиться хотя бы 0.9 F1 на тестовой выборке. Заметьте, что чем выше качество большой модели, тем лучше будет работать дистиллированный ученик. Для обучения можно использовать `Trainer` из Hugging Face.

In [ ]:
seed_everything()
model = e.make_teacher()
print('Число параметров:', sum(p.numel() for p in model.parameters()))


In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=2.0):
    mask = labels != -100
    student_log_probs = F.log_softmax(student_logits[mask].float() / temperature, dim=-1)
    teacher_probs = F.softmax(teacher_logits[mask].float() / temperature, dim=-1)
    return F.kl_div(student_log_probs, teacher_probs, reduction='batchmean') * temperature**2


def train(stage, model, data, epochs, batch_size, lr, max_steps, patience, teacher=None):
    folder = OUTPUT / stage
    folder.mkdir(parents=True, exist_ok=True)
    count = sum(p.numel() for p in model.parameters())
    print(f'{stage}: {count:,} parameters on {DEVICE}', flush=True)
    train_loader = data['train']
    steps = min(max_steps, epochs * len(train_loader)) if max_steps else epochs * len(train_loader)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(steps * 0.1)),
        num_training_steps=steps,
    )
    if teacher is not None:
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
    best, bad_epochs, step = -1.0, 0, 0
    history = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        epoch_steps = 0
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=DEVICE.type == 'cuda'):
                out = model(**batch)
                loss = out.loss
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = teacher(**batch).logits
                    loss = loss + distillation_loss(out.logits, teacher_logits, batch['labels'])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            step += 1
            epoch_steps += 1
            if step % 100 == 0:
                print(f'{stage} step={step}/{steps} loss={loss.item():.4f}', flush=True)
            if step >= steps:
                break
        valid = evaluate(model, data['validation'])
        record = {
            'epoch': epoch + 1,
            'step': step,
            'train_loss': total_loss / epoch_steps,
            'validation_f1': valid['f1'],
        }
        history.append(record)
        print(json.dumps(record), flush=True)
        (folder / 'history.json').write_text(json.dumps(history, indent=2))
        if valid['f1'] > best:
            best, bad_epochs = valid['f1'], 0
            torch.save(model.state_dict(), folder / 'best.pt')
        else:
            bad_epochs += 1
        if step >= steps or bad_epochs >= patience:
            break
    load_state(model, stage)
    test = evaluate(model, data['test'])
    summary = {
        'parameters': count,
        'best_validation_f1': best,
        'test_f1': test['f1'],
        'test_report': test['report'],
        'steps': step,
    }
    (folder / 'metrics.json').write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2), flush=True)
    return summary


def scores(stage):
    folder = OUTPUT / stage
    return {
        name: json.loads((folder / name).read_text())
        for name in ['metrics.json', 'word_metrics.json']
        if (folder / name).exists()
    }

if not (OUTPUT / 'teacher' / 'best.pt').exists():
    data = loaders(tokenizer, tokenized_dataset, 32)
    train('teacher', model, data, 5, 32, 3e-5, 0, 3)
model = load_state(model, 'teacher')
scores('teacher')


### Факторизация матрицы эмбеддингов

Можно заметить, что на данный момент матрица эмбеддингов занимает $V \cdot H = 28996 \cdot 768 = 22.268.928$ параметров. Это aж пятая часть от всей модели! Давайте попробуем что-то с этим сделать. В модели [ALBERT](https://arxiv.org/pdf/1909.11942.pdf) предлагается факторизовать матрицу эмбеддингов в произведение двух небольших матриц. Таким образом, параметры эмбеддингов будут содержать $V \cdot E + E \cdot H$ элементов, что гораздо меньше $V \cdot H$, если $H \gg E$. Авторы выбирают $E = 128$, однако ничего не мешает нам взять любое другое значение. Например, выбрав $H = 64$, мы уменьшим число параметров примерно на 20М.

__Задание 3 (1 балл).__ Напишите класс-обертку над слоем эмбеддингов, который реализует факторизацию на две матрицы, и дообучите факторизованную модель. Заметьте, обе матрицы можно инициализировать с помощью SVD разложения (а лучше [FWSVD](https://arxiv.org/pdf/2207.00112)!), чтобы начальное приближение было хорошим. Это сэкономит очень много времени на дообучении. С рангом разложения $H = 64$ у вас должно получиться F1 больше 0.87.

In [ ]:
class FactorizedEmbedding(nn.Module):
    def __init__(self, matrix, rank=64, padding_idx=0):
        super().__init__()
        with torch.no_grad():
            u, s, v = torch.svd_lowrank(matrix.detach().float(), q=rank + 16, niter=4)
            s = s[:rank].sqrt()
            left = u[:, :rank] * s
            right = s[:, None] * v[:, :rank].T
        self.embedding = nn.Embedding.from_pretrained(left, freeze=False, padding_idx=padding_idx)
        self.projection = nn.Linear(rank, matrix.shape[1], bias=False, device=matrix.device)
        self.projection.weight.data.copy_(right.T)

    def forward(self, input_ids):
        return self.projection(self.embedding(input_ids))


if not (OUTPUT / 'factorized' / 'best.pt').exists():
    factorized = load_state(e.make_teacher(), 'teacher')
    embeddings = factorized.bert.embeddings.word_embeddings
    factorized.bert.embeddings.word_embeddings = FactorizedEmbedding(
        embeddings.weight, rank=64, padding_idx=tokenizer.pad_token_id
    )
    data = loaders(tokenizer, tokenized_dataset, 32)
    train('factorized', factorized, data, 5, 32, 2e-5, 0, 3)
scores('factorized')


### Дистилляция знаний

Дистилляция знаний – это парадигма обучения, в которой знания модели-учителя дистиллируются в модель-ученика. Учеником может быть произвольная модель меньшего размера, решающая ту же задачу, однако обычно ученик имеет ту же архитектуру, что и учитель. При дистилляции используются два функционала ошибки:

1. Стандартная кросс-энтропия.
1. Функция, задающая расстояние между распределениями предсказаний учителя и ученика. Чаще всего используют KL-дивергенцию.

Для того, чтобы распределение предсказаний учителя не было вырожденным, к softmax добавляют температуру больше 1, например, 2 или 5.   
__Важно:__ при делении логитов на температуру значения градиентов уменьшаются в $\tau^2$ раз (проверьте это!). Поэтому для возвращения их в изначальный масштаб ошибку надо домножить на $\tau^2$. Подробнее об этом можно почитать в разделе 2.1 [оригинальной статьи](https://arxiv.org/pdf/1503.02531).

<img src="https://intellabs.github.io/distiller/imgs/knowledge_distillation.png" width="800">

__Задание 4 (3 балла).__ Реализуйте метод дистилляции знаний, изображенный на картинке. Для подсчета ошибки между предсказаниями ученика и учителя используйте KL-дивергенцию [`nn.KLDivLoss(reduction="batchmean")`](https://pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html) (обратите внимание на вормат ее входов). Для получения итоговой ошибки суммируйте мягкую ошибку с жесткой.   
В качестве учителя используйте дообученный BERT из задания 2. В качестве ученика возьмите необученную модель с размером __не больше 20M__ параметров. Вы можете использовать факторизацию матрицы эмбеддингов для уменьшения числа параметров. Если вы все сделали правильно, то на тестовой выборке вы должны получить значение F1 не меньше 0.7. Вам должно хватить примерно 20к итераций обучения для этого. Если у вас что-то не получается, то можно ориентироваться на статью про [DistilBERT](https://arxiv.org/abs/1910.01108) и на [эту статью](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT).

__Важно:__
* Не забывайте добавлять _warmup_ при обучении ученика.
* Не забывайте переводить учителя в режим _eval_.

In [ ]:
def make_student(tokenizer):
    config = BertConfig(
        vocab_size=tokenizer.vocab_size,
        hidden_size=384,
        num_hidden_layers=4,
        num_attention_heads=6,
        intermediate_size=1536,
        num_labels=len(LABELS),
        pad_token_id=tokenizer.pad_token_id,
    )
    model = BertForTokenClassification(config).to(DEVICE)
    assert sum(p.numel() for p in model.parameters()) <= 20_000_000
    return model


if not (OUTPUT / 'student' / 'best.pt').exists():
    student_model = make_student(tokenizer)
    data = loaders(tokenizer, tokenized_dataset, 64)
    train('student', student_model, data, 100, 64, 3e-4, 20000, 8, teacher=model)
scores('student')


# Задания на выбор

Как вы понимаете, есть еще довольно много разных способов уменьшить обученную модель. В этой секции вам предлагается реализовать разные техники на выбор. За каждую из них можно получить разное количество балов в зависимости от сложности. Успешность реализации будет оцениваться как по коду, так и по качеству на тестовой выборке. Все баллы за это дз, которые вы наберете сверх 10, будут считаться бонусными.   
В задании 4 вы обучали модель с ограничением числа параметров в 20М. При реализации техник из этой секции придерживайтесь такого же ограничения. Это позволит честно сравнивать методы между собой и делать правильные выводы. Напишите в отчете обо всем, что вы попробовали.

* __Шеринг весов (1 балл).__ В модификации BERT [ALBERT](https://arxiv.org/pdf/1909.11942) помимо факторизации эмбеддингов предлагается шерить веса между слоями. То есть разные слои используют одни и те же веса. Такая техника эвивалентна применению одного и того же слоя несколько раз. Она позволяет в несколько раз уменьшить число параметров и не сильно потерять в качестве.
* __Шеринг весов++ (0.5 балл).__ Вместо того, чтобы просто применять одни и те же слои, можно дообучить LoRA для каждого слоя, чтобы слои могли специализироваться. Ровно так делают в [этой статье](https://arxiv.org/pdf/2410.20672) от DeepMind.
* __Приближение промежуточных слоев (1.5 балла).__ Мы обсуждали, что помимо приближения выходов модели ученика к выходам модели учителя, можно приближать выходы промежуточных слоев. В [этой работе](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT) подробно написано, как это можно сделать. Еще __+1 балл__ за реализацию метода из [MiniLMv2](https://arxiv.org/pdf/2012.15828).
* __Прунинг (3 балла).__ В методе [SparseGPT](https://arxiv.org/abs/2301.00774) предлагается подход, удаляющий веса модели один раз после обучения. При этом оказывается возможным удалить до половины всех весов без потери в качестве. Математика, стоящаяя за техникой, довольно сложная, однако общий подход простой – будем удалять веса в каждом слое по отдельности, при удалении части весов слоя, остальные веса будут перенастраиваться так, чтобы общий выход слоя не изменился. В качестве бейзлайна реализуйте более простой метод [Wanda](https://arxiv.org/pdf/2306.11695).
* __Инициализация ученика прунингом (2 балла).__ Обучение ученика с нуля занимает достаточно много времени. Вместо этого, ученика можно инициализоровать с помощью прунинга учителя, а затем дообучить с помощью дистилляции. Впервые так сделали в [Minitron](https://arxiv.org/pdf/2408.11796), а после в Ministral 3, Llama 3.2, Falcon3 и т.д. Важная проблема этого подхода заключается в том, что слои могут пруниться неравномерно, из-за чего при использовании модели будет сложнее распределять нагрузку. Но если задача – просто уменьшить размер, то это отличный способ.



In [ ]:
class LayerWithAdapter(nn.Module):
    def __init__(self, layer, hidden_size=384, rank=16):
        super().__init__()
        self.layer = layer
        self.down = nn.Linear(hidden_size, rank, bias=False)
        self.up = nn.Linear(rank, hidden_size, bias=False)
        nn.init.zeros_(self.up.weight)

    def forward(self, *args, **kwargs):
        inputs = args[0] if args else kwargs['hidden_states']
        result = self.layer(*args, **kwargs)
        hidden = result[0]
        return (hidden + self.up(self.down(inputs)), *result[1:])


def shared_student(tokenizer, adapters=False):
    student = e.make_student(tokenizer)
    shared = student.bert.encoder.layer[0]
    if adapters:
        layers = [LayerWithAdapter(shared) for _ in range(4)]
    else:
        layers = [shared for _ in range(4)]
    student.bert.encoder.layer = nn.ModuleList(layers).to(e.DEVICE)
    return student


def pruned_student(tokenizer, teacher):
    student = e.make_student(tokenizer)
    source = teacher.state_dict()
    selected = {}
    for name, target in student.state_dict().items():
        parts = name.split('.')
        if len(parts) > 3 and parts[:3] == ['bert', 'encoder', 'layer']:
            parts[3] = str((int(parts[3]) + 1) * 3 - 1)
        original = source['.'.join(parts)]
        slices = tuple(slice(0, size) for size in target.shape)
        value = original[slices]
        if value.shape != target.shape:
            raise ValueError(f'Cannot prune {name}: {original.shape} -> {target.shape}')
        selected[name] = value.clone()
    student.load_state_dict(selected)
    return student


def train_intermediate(student, teacher, data, batch_size, epochs, max_steps):
    stage = 'intermediate'
    folder = e.OUTPUT / stage
    folder.mkdir(parents=True, exist_ok=True)
    student.projections = nn.ModuleList([nn.Linear(384, 768).to(e.DEVICE) for _ in range(4)])
    count = sum(p.numel() for p in student.parameters())
    assert count <= 20_000_000
    teacher.eval()
    for param in teacher.parameters():
        param.requires_grad_(False)
    steps = min(max_steps, epochs * len(data['train']))
    optimizer = AdamW(student.parameters(), lr=3e-4, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, max(1, int(steps * 0.1)), steps)
    best, bad_epochs, step = -1.0, 0, 0
    history = []
    print(f'{stage}: {count:,} parameters on {e.DEVICE}', flush=True)
    for epoch in range(epochs):
        student.train()
        losses = []
        for batch in data['train']:
            batch = {k: v.to(e.DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=e.DEVICE.type == 'cuda'):
                out = student(**batch, output_hidden_states=True)
                with torch.no_grad():
                    target = teacher(**batch, output_hidden_states=True)
                mask = batch['labels'].ne(-100)
                soft = e.distillation_loss(out.logits, target.logits, batch['labels'])
                hidden = sum(
                    F.mse_loss(
                        student.projections[i](out.hidden_states[i + 1])[mask].float(),
                        target.hidden_states[(i + 1) * 3][mask].float(),
                    )
                    for i in range(4)
                ) / 4
                loss = out.loss + soft + hidden
            loss.backward()
            nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            losses.append(loss.item())
            step += 1
            if step % 100 == 0:
                print(f'{stage} step={step}/{steps} loss={loss.item():.4f}', flush=True)
            if step >= steps:
                break
        valid = e.evaluate(student, data['validation'])
        history.append({
            'epoch': epoch + 1,
            'step': step,
            'train_loss': sum(losses) / len(losses),
            'validation_f1': valid['f1'],
        })
        print(json.dumps(history[-1]), flush=True)
        (folder / 'history.json').write_text(json.dumps(history, indent=2))
        if valid['f1'] > best:
            best, bad_epochs = valid['f1'], 0
            torch.save(student.state_dict(), folder / 'best.pt')
        else:
            bad_epochs += 1
        if step >= steps or bad_epochs >= 8:
            break
    e.load_state(student, stage)
    test = e.evaluate(student, data['test'])
    summary = {
        'parameters': count,
        'best_validation_f1': best,
        'test_f1': test['f1'],
        'test_report': test['report'],
        'steps': step,
    }
    (folder / 'metrics.json').write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2), flush=True)


@torch.inference_mode()
def collect_statistics(model, loader, batches):
    layers = {
        name: module
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear) and name.startswith('bert.encoder.layer')
    }
    statistics = {name: {'sum': None, 'gram': None, 'count': 0} for name in layers}
    hooks = []

    def hook(name):
        def record(module, inputs, output):
            values = inputs[0].detach().reshape(-1, inputs[0].shape[-1]).float()
            item = statistics[name]
            squared = values.square().sum(dim=0)
            gram = values.T @ values
            item['sum'] = squared if item['sum'] is None else item['sum'] + squared
            item['gram'] = gram if item['gram'] is None else item['gram'] + gram
            item['count'] += values.shape[0]
        return record

    for name, layer in layers.items():
        hooks.append(layer.register_forward_hook(hook(name)))
    model.eval()
    for i, batch in enumerate(loader):
        if i >= batches:
            break
        batch = {k: v.to(e.DEVICE) for k, v in batch.items() if k != 'labels'}
        model(**batch)
    for handle in hooks:
        handle.remove()
    return layers, statistics


def row_mask(scores, sparsity):
    count = int(scores.shape[1] * sparsity)
    mask = torch.zeros_like(scores, dtype=torch.bool)
    if count:
        mask.scatter_(1, scores.argsort(dim=1)[:, :count], True)
    return mask


@torch.no_grad()
def wanda(layer, stats, sparsity):
    weight = layer.weight.data
    activation = (stats['sum'] / stats['count']).sqrt()
    weight[row_mask(weight.abs() * activation.unsqueeze(0), sparsity)] = 0


@torch.no_grad()
def sparsegpt(layer, stats, sparsity, block_size=128):
    weight = layer.weight.data.float().clone()
    hessian = stats['gram'] / stats['count']
    diagonal = hessian.diagonal()
    dead = diagonal == 0
    diagonal[dead] = 1
    weight[:, dead] = 0
    diagonal.add_(0.01 * diagonal.mean())
    inverse = torch.cholesky_inverse(torch.linalg.cholesky(hessian))
    inverse = torch.linalg.cholesky(inverse, upper=True)
    for start in range(0, weight.shape[1], block_size):
        end = min(start + block_size, weight.shape[1])
        current = weight[:, start:end].clone()
        block_inverse = inverse[start:end, start:end]
        scores = current.square() / block_inverse.diagonal().square().unsqueeze(0)
        mask = row_mask(scores, sparsity)
        errors = torch.zeros_like(current)
        for column in range(end - start):
            value = current[:, column]
            quantized = value.clone()
            quantized[mask[:, column]] = 0
            error = (value - quantized) / block_inverse[column, column]
            current[:, column:] -= error.unsqueeze(1) * block_inverse[column, column:].unsqueeze(0)
            current[:, column] = quantized
            errors[:, column] = error
        weight[:, start:end] = current
        if end < weight.shape[1]:
            weight[:, end:] -= errors @ inverse[start:end, end:]
    layer.weight.data.copy_(weight.to(layer.weight.dtype))


def capture_relations(model, storage):
    attention = model.bert.encoder.layer[-1].attention.self
    handles = []
    for name in ('query', 'key', 'value'):
        def capture(module, inputs, output, key=name):
            storage[key] = output
        handles.append(getattr(attention, name).register_forward_hook(capture))
    return handles


def relation_loss(student, teacher, attention_mask, heads=12):
    valid = attention_mask.bool()
    total = 0
    for name in ('query', 'key', 'value'):
        relations = []
        for states in (student[name], teacher[name]):
            batch, length, width = states.shape
            vectors = states.float().reshape(batch, length, heads, width // heads).transpose(1, 2)
            scores = vectors @ vectors.transpose(-1, -2) / math.sqrt(width // heads)
            relations.append(scores.masked_fill(~valid[:, None, None, :], -1e4))
        log_prob = F.log_softmax(relations[0], dim=-1)
        target = F.softmax(relations[1], dim=-1)
        per_query = F.kl_div(log_prob, target, reduction='none').sum(-1)
        total += (per_query * valid[:, None, :]).sum() / (valid.sum() * heads)
    return total / 3


{stage: scores(stage) for stage in [
    'pruned_init', 'sharing', 'sharing_lora', 'intermediate',
    'minilmv2', 'wanda', 'sparsegpt',
]}
